In [320]:
import polars as pl
import re

# citation-claims

In [321]:
claims = pl.read_parquet("output/curator_claims.parquet")
claims.head()

gene_id,sentence_markers,sentence_plain,cited_sentence_marked,claim_plain,anchors,publication_ids,citation_captions,citation_years
str,str,str,str,str,list[struct[2]],list[i64],list[str],list[i64]
"""DDB_G0287681""","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","""grlR encodes a member of the e…","[{126,[827]}]",[827],"[""Prabu and Eichinger 2006)""]",[2006]
"""DDB_G0290825""","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","""The polyketide synthase StlB (…","[{144,[956]}, {253,[2919]}]","[956, 2919]","[""(Austin et al. 2006)"", ""(Thompson and Kay, 2000)""]","[2006, 2000]"
"""DDB_G0285319""","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","""There are three histone H3prot…","[{39,[4032]}]",[4032],"[""Bukenberger et al. 1992""]",[1992]
"""DDB_G0292810""","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","""Analyses of gluA null mutants …","[{112,[9912]}]",[9912],"[""Dimond and Loomis 1976)""]",[1976]
"""DDB_G0292206""","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","""Expression is dependent on a s…","[{126,[6149]}]",[6149],"[""Schatzle et al. 1991)""]",[1991]


In [322]:
# look at a row closely
row = claims.row(1, named=True)
print(row)

{'gene_id': 'DDB_G0290825', 'sentence_markers': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [[PUB:956]] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [[PUB:2919]].', 'sentence_plain': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) (Austin et al. 2006) and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH (Thompson and Kay, 2000).', 'cited_sentence_marked': 'The polyketide synthase StlB (stlB) catalyzes the first step, the formation of the core polyketide (2,4,6-trihydroxyphenyl)-1-hexan-1-one (THPH) [CITE:956] and des-methyl-DIF-1 methyltransferase DmtA (dmtA) catalyzes the last step, the methylation of dichloro-THPH [CITE:2919].', 'claim_plain': 'The polyketide synthase S

In [323]:
# get preferred column set (same as before)
df = claims.select([
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id",
])

In [324]:
# clean up parentheses in citation_captions
df = df.with_columns(
    pl.col("citation_captions")
    .list.eval(
        pl.element()
        .str.replace_all(r"[()]", "")   # remove parentheses
        .str.replace_all(r"\s+", " ")   # collapse whitespace
        .str.strip_chars()              # <-- instead of .str.strip()
    )
    .alias("citation_captions")
)
# clean up comma
df = df.with_columns(
    pl.col("citation_captions")
    .list.eval(
        pl.element()
        .str.replace_all(r"[()]", "")
        .str.replace_all(r"\bet al\b\.?", "et al.")
        .str.replace_all(r",\s*et al\.", " et al.")
        .str.replace_all(r"et al\.\s*,\s*", "et al. ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )
    .alias("citation_captions")
)

In [325]:
# clean up very short claims, it shall have at least 6
min_words = 6

df2 = df.with_columns(
    # Count "words" by matching non-space sequences
    pl.col("claim_plain")
      .str.count_matches(r"\S+")
      .alias("n_words")
)

# 1) summary
summary = df2.select([
    pl.len().alias("rows_total"),
    (pl.col("n_words") < min_words).sum().alias("rows_to_drop"),
    (pl.col("n_words") >= min_words).sum().alias("rows_to_keep"),
])
print(summary)
# 2) inspect what you'd drop (optional)
to_drop = (
    df2.filter(pl.col("n_words") < min_words)
       .select(["n_words", "claim_plain", "gene_id"])
       .sort(["n_words", "claim_plain"])
)
with pl.Config(fmt_str_lengths=300):
    display(to_drop.head(50))

# 3) actually filter
df_filtered = df2.filter(pl.col("n_words") >= min_words).drop("n_words")

shape: (1, 3)
┌────────────┬──────────────┬──────────────┐
│ rows_total ┆ rows_to_drop ┆ rows_to_keep │
│ ---        ┆ ---          ┆ ---          │
│ u32        ┆ u32          ┆ u32          │
╞════════════╪══════════════╪══════════════╡
│ 2677       ┆ 24           ┆ 2653         │
└────────────┴──────────────┴──────────────┘


n_words,claim_plain,gene_id
u32,str,str
1,""".""","""DDB_G0284845"""
1,"""pneumoniae.""","""DDB_G0267444"""
1,"""pneumoniae.""","""DDB_G0267630"""
1,"""pneumoniae.""","""DDB_G0279183"""
2,"""(gskA) (..""","""DDB_G0281385"""
…,…,…
5,"""STATa is downregulated by PTP1.""","""DDB_G0281381"""
5,"""Spores have lower cellulose levels.""","""DDB_G0281387"""
5,"""at the tipped aggregate stage.""","""DDB_G0286185"""


In [326]:
# let us examine the duplicated claims closely
# 1) merge duplicates by concatenating gene_id (unique + sorted)
merged = (
    df_filtered
    .group_by([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .agg(
        pl.col("gene_id")
          .unique()
          .sort()
          .str.join(",")
          .alias("gene_id")
    )
    .select([
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions",
        "gene_id",
    ])
)

print("after merge:")
print("rows =", merged.height)
print("unique claim_plain =", merged.select(pl.col("claim_plain").n_unique()).item())

# 2) check if claim_plain is STILL duplicated (i.e., same claim text but different citation/anchors/etc.)
still_dups = (
    merged
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain"])
)

print("still duplicated claim_plain rows =", still_dups.height)

# show them grouped together if any remain
still_dups

after merge:
rows = 2107
unique claim_plain = 2103
still duplicated claim_plain rows = 8


claim_plain,anchors,publication_ids,citation_captions,gene_id,n
str,list[struct[2]],list[i64],list[str],str,u32
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser et al. 1995""]","""DDB_G0267402,DDB_G0270838,DDB_…",2
"""In Dictyostelium discoideum an…","[{183,[5046]}]",[5046],"[""Hauser and colleagues 1995""]","""DDB_G0285319""",2
"""The 2C gene as well as 5 relat…","[{149,[10129]}]",[10129],"[""Schilde et al. 2004""]","""DDB_G0280871,DDB_G0280953,DDB_…",2
"""The 2C gene as well as 5 relat…","[{149,[1381]}]",[1381],"[""Schilde et al. 2004""]","""DDB_G0280847""",2
"""These mutant cells are also im…","[{148,[2842]}]",[2842],"[""Wessels et al. 2000""]","""DDB_G0284331""",2
"""These mutant cells are also im…","[{148,[1548]}]",[1548],"[""Zhang et al. 2003""]","""DDB_G0279413""",2
"""abcF1 and abcF4 are the most c…","[{115,[2279]}]",[2279],"[""Anjard and Loomis 2002""]","""DDB_G0285997""",2
"""abcF1 and abcF4 are the most c…","[{115,[8189]}]",[8189],"[""Anjard and Loomis 2002""]","""DDB_G0267436,DDB_G0275637,DDB_…",2


In [327]:
# with pl.Config(fmt_str_lengths=60):
#     display(still_dups)       


There are some duplicated claims.

Except the first one, the others are claims with inconsistant citations, let us remove those except the first one.

In [328]:
# 1) add a stable row id to merged
merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")

# 2) recompute still_dups from merged_i (so it contains row_nr)
still_dups_i = (
    merged_i
    .with_columns(pl.len().over("claim_plain").alias("n"))
    .filter(pl.col("n") > 1)
    .sort(["claim_plain", "row_nr"])
)

# 3) drop everything in still_dups except the first row of still_dups
drop_row_nrs = still_dups_i.slice(1).select("row_nr")   # everything after the first row

merged_clean = (
    merged_i
    .join(drop_row_nrs, on="row_nr", how="anti")  # remove those rows
    .drop(["row_nr", "n"], strict=False)
)


/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_50959/3282369987.py:2: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  merged_i = merged.with_row_count("row_nr")   # if this errors on your Polars, use: .with_row_index("row_nr")


In [329]:
merged_clean

claim_plain,anchors,publication_ids,citation_captions,gene_id
str,list[struct[2]],list[i64],list[str],str
"""It was shown that MHCK A activ…","[{147,[6393]}]",[6393],"[""Medley et al. 1990""]","""DDB_G0291231"""
"""The combination of a myosin mo…","[{197,[3083]}]",[3083],"[""Geissler et al. 2000""]","""DDB_G0292262"""
"""However, Nhe1 has been shown t…","[{159,[13]}]",[13],"[""Lusche et al. 2011""]","""DDB_G0275711"""
"""RasG also regulates the plasma…","[{111,[824]}, {114,[1464]}, … {120,[17325]}]","[824, 1464, … 17325]","[""Bolourani et al. 2006"", ""Sasaki et al. 2004"", … ""Baumgardner et al. 2018""]","""DDB_G0293434"""
"""These results indicate that ra…","[{178,[286]}]",[286],"[""Parkinson et al. 2009""]","""DDB_G0282247"""
…,…,…,…,…
"""This could explain why alrA - …","[{53,[1321]}]",[1321],"[""Ehrenman et al. 2004""]","""DDB_G0293850"""
"""TagC is a composite protein in…","[{90,[4883]}]",[4883],"[""Shaulsky et al. 1995""]","""DDB_G0286121"""
"""Measurements of the specific a…","[{225,[10645]}]",[10645],"[""Loomis et al. 1969""]","""DDB_G0287033"""


In [330]:
# give claim ID, and we want to explode the list columns together to get one row per (claim_id, publication_id)
m = merged_clean.with_columns(
    pl.col("claim_plain").rank(method="dense").cast(pl.Int64).alias("claim_id")
).select([
    "claim_id",
    "claim_plain",
    "anchors",
    "publication_ids",
    "citation_captions",
    "gene_id"
])

In [332]:
# 2) sanity check: list columns must be aligned per row before explode (len(citation_captions) == len(publication_ids))
bad_rows = (
    m
    .with_columns([
        pl.col("publication_ids").list.len().alias("n_pub"),
        pl.col("citation_captions").list.len().alias("n_cap")
    ])
    .filter(
        (pl.col("n_pub") != pl.col("n_cap")) 
    )
    .select([
        "claim_id",
        "gene_id",
        "n_pub", "n_cap", 
        "claim_plain",
        "anchors",
        "publication_ids",
        "citation_captions"
    ])
    .sort(["n_pub", "n_cap"], descending=True)
)

print("Number of misaligned rows:", bad_rows.height)

with pl.Config(fmt_str_lengths=1000):
    display(bad_rows)

Number of misaligned rows: 4


claim_id,gene_id,n_pub,n_cap,claim_plain,anchors,publication_ids,citation_captions
i64,str,u32,u32,str,list[struct[2]],list[i64],list[str]
1469,"""DDB_G0293084""",2,1,"""The zizB mutant phenotypes and ZizB binding partners suggest a central role for ZizB in actin cytoskeletal organization and cortical stabilization.,.""","[{147,[12098]}, {148,[12830]}]","[12098, 12830]","[""Pakes et al. 2012""]"
471,"""DDB_G0268620""",2,1,"""Fluid uptake by macropinocytosis is reduced about 3 fold and phagosomes remain more acidic in pkbA - null cells (, (.""","[{113,[2428]}, {116,[2634]}]","[2428, 2634]","[""Rupper et al. 2001""]"
1534,"""DDB_G0287031""",2,1,"""These results reveal that gpaC functions as a regulator of early development gene expression,.""","[{92,[4035]}, {93,[4034]}]","[4035, 4034]","[""Brandon et al. 1997""]"
1195,"""DDB_G0286183""",2,1,"""The Dictyostelium Agps protein has been crystallized.""","[{52,[712, 659]}]","[712, 659]","[""Razeto et al. ., 2007""]"


in the 4 cases here, len(publication_ids)>len(citation_captions), after closer look, it is the same author and same year, we manully make a patch here

In [333]:
m_fixed = (
    m.with_columns(
        pl.when(pl.col("claim_id") == 1534)
          .then(pl.lit(["Brandon et al. 1997a", "Brandon et al. 1997b"]))
        .when(pl.col("claim_id") == 471)
          .then(pl.lit(["Rupper et al. 2001a", "Rupper et al. 2001b"]))
        .when(pl.col("claim_id") == 1469)
          .then(pl.lit(["Pakes et al. 2012a", "Pakes et al. 2012b"]))
        .when(pl.col("claim_id") == 1195)
          .then(pl.lit(["Razeto et al. 2007a", "Razeto et al. 2007b"]))
        .otherwise(pl.col("citation_captions"))
        .alias("citation_captions")
    )
)

In [334]:
m_long = (
    m_fixed
    .explode(["publication_ids", "citation_captions"])
    .rename({"publication_ids": "publication_id"})
)

In [335]:
m_long

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id
i64,str,list[struct[2]],i64,str,str
793,"""It was shown that MHCK A activ…","[{147,[6393]}]",6393,"""Medley et al. 1990""","""DDB_G0291231"""
1276,"""The combination of a myosin mo…","[{197,[3083]}]",3083,"""Geissler et al. 2000""","""DDB_G0292262"""
578,"""However, Nhe1 has been shown t…","[{159,[13]}]",13,"""Lusche et al. 2011""","""DDB_G0275711"""
1076,"""RasG also regulates the plasma…","[{111,[824]}, {114,[1464]}, … {120,[17325]}]",824,"""Bolourani et al. 2006""","""DDB_G0293434"""
1076,"""RasG also regulates the plasma…","[{111,[824]}, {114,[1464]}, … {120,[17325]}]",1464,"""Sasaki et al. 2004""","""DDB_G0293434"""
…,…,…,…,…,…
1587,"""This could explain why alrA - …","[{53,[1321]}]",1321,"""Ehrenman et al. 2004""","""DDB_G0293850"""
1149,"""TagC is a composite protein in…","[{90,[4883]}]",4883,"""Shaulsky et al. 1995""","""DDB_G0286121"""
832,"""Measurements of the specific a…","[{225,[10645]}]",10645,"""Loomis et al. 1969""","""DDB_G0287033"""


In [336]:
# now we exlpode anchors
# 1) Build mapping: (claim_id, publication_id) -> anchor_pos
anchor_map = (
    m_fixed
    .select(["claim_id", "anchors"])
    .explode("anchors")  # now each row is one struct {pos, pub_ids}
    .with_columns([
        pl.col("anchors").struct.field("pos").alias("anchor_pos"),
        pl.col("anchors").struct.field("pub_ids").alias("publication_id"),
    ])
    .explode("publication_id")  # one row per pub_id
    .select(["claim_id", "publication_id", "anchor_pos"])
)

# If there can be multiple anchor_pos for the same (claim_id, publication_id), keep them all:
anchor_map = (
    anchor_map
    .group_by(["claim_id", "publication_id"])
    .agg(pl.col("anchor_pos").sort().alias("anchor_pos"))
)

# 2) Join onto m_long
m_long2 = m_long.join(anchor_map, on=["claim_id", "publication_id"], how="left")

# 3) If you want exactly one row per (claim_id, publication_id, anchor_pos), explode anchor_pos:
m_long3 = m_long2.explode("anchor_pos")

# Final columns (example)
m_long3.select([
    "claim_id",
    "claim_plain",
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
])

claim_id,claim_plain,anchor_pos,publication_id,citation_captions,gene_id
i64,str,i64,i64,str,str
793,"""It was shown that MHCK A activ…",147,6393,"""Medley et al. 1990""","""DDB_G0291231"""
1276,"""The combination of a myosin mo…",197,3083,"""Geissler et al. 2000""","""DDB_G0292262"""
578,"""However, Nhe1 has been shown t…",159,13,"""Lusche et al. 2011""","""DDB_G0275711"""
1076,"""RasG also regulates the plasma…",111,824,"""Bolourani et al. 2006""","""DDB_G0293434"""
1076,"""RasG also regulates the plasma…",114,1464,"""Sasaki et al. 2004""","""DDB_G0293434"""
…,…,…,…,…,…
1587,"""This could explain why alrA - …",53,1321,"""Ehrenman et al. 2004""","""DDB_G0293850"""
1149,"""TagC is a composite protein in…",90,4883,"""Shaulsky et al. 1995""","""DDB_G0286121"""
832,"""Measurements of the specific a…",225,10645,"""Loomis et al. 1969""","""DDB_G0287033"""


In [337]:
# now we get year from citation_captions
YEAR_RE = r"(18|19|20)\d{2}"

m_long3 = m_long3.with_columns(
    year=pl.col("citation_captions")
        .str.extract(YEAR_RE, 0)   # whole match
        .cast(pl.Int32)
)
m_long3

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year
i64,str,list[struct[2]],i64,str,str,i64,i32
793,"""It was shown that MHCK A activ…","[{147,[6393]}]",6393,"""Medley et al. 1990""","""DDB_G0291231""",147,1990
1276,"""The combination of a myosin mo…","[{197,[3083]}]",3083,"""Geissler et al. 2000""","""DDB_G0292262""",197,2000
578,"""However, Nhe1 has been shown t…","[{159,[13]}]",13,"""Lusche et al. 2011""","""DDB_G0275711""",159,2011
1076,"""RasG also regulates the plasma…","[{111,[824]}, {114,[1464]}, … {120,[17325]}]",824,"""Bolourani et al. 2006""","""DDB_G0293434""",111,2006
1076,"""RasG also regulates the plasma…","[{111,[824]}, {114,[1464]}, … {120,[17325]}]",1464,"""Sasaki et al. 2004""","""DDB_G0293434""",114,2004
…,…,…,…,…,…,…,…
1587,"""This could explain why alrA - …","[{53,[1321]}]",1321,"""Ehrenman et al. 2004""","""DDB_G0293850""",53,2004
1149,"""TagC is a composite protein in…","[{90,[4883]}]",4883,"""Shaulsky et al. 1995""","""DDB_G0286121""",90,1995
832,"""Measurements of the specific a…","[{225,[10645]}]",10645,"""Loomis et al. 1969""","""DDB_G0287033""",225,1969


In [338]:
m_long4 = m_long3.select([
    "claim_id",
    "claim_plain",
    "anchors",        # keep if you still want the original struct list
    "anchor_pos",
    "publication_id",
    "citation_captions",
    "gene_id",
    "year",
])

i still see some claims needs cleaned up. They are almost identical with only dot diffrenece.


In [339]:
claim_key_expr = (
    pl.col("claim_plain")
    .str.to_lowercase()
    .str.replace_all(r"\.{2,}", ".")          # ".." / "..." -> "."
    .str.replace_all(r"[^a-z0-9\s]", " ")     # drop punctuation
    .str.replace_all(r"\s+", " ")             # collapse whitespace
    .str.strip_chars()
)

tmp = m_long4.with_columns(
    claim_key=claim_key_expr
)

# Optional: see groups where multiple raw claim_plain map to same key
near_same = (
    tmp.group_by("claim_key")
       .agg([
           pl.len().alias("n_rows"),
           pl.col("claim_plain").n_unique().alias("n_variants"),
           pl.col("claim_plain"),
       ])
       .filter(pl.col("n_variants") > 1)
       .sort("n_rows", descending=True)
)
near_same

claim_key,n_rows,n_variants,claim_plain
str,u32,u32,list[str]
"""at the trailing edge activated…",9,2,"[""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,.."", ""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,.."", … ""At the trailing edge activated Rac1 forms a complex with DGAP1 and the actin-binding proteins cortexillin I (ctxA) and cortexillin II (ctxB) enabling proper cell retraction,,,,.""]"
"""the major spore coat proteins …",9,2,"[""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;;,."", ""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;;,."", … ""The major spore coat proteins, SP96, SP70, and SP60 ( cotA, cotB, cotC, respectively), are coordinately synthesized in prespore cells and stored in prespore vesicles (PSVs);;,.""]"
"""dictyostelium has three genes …",6,3,"[""Dictyostelium has three genes encoding profilin, proA, (proB) and (proC),."", ""Dictyostelium has three genes encoding profilin, proA, (proB) and (proC),."", … ""Dictyostelium has three genes encoding profilin, (proA), (proB) and proC,.""]"
"""expression of cota sp96 cotb s…",6,2,"[""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;."", ""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;."", … ""Expression of cotA (SP96), cotB (SP70) and cotC (SP60) has been used to monitor prespore specific transcription in a large number of studies (e.g.,;;..""]"
"""about 100 different proteins c…",6,6,"[""About 100 different proteins contribute to centrosomal functions including the products of (tubC), (spc97), (spc98), (cenA), lis1, (nek2), (eb1), (cepJ) and (cepG)."", ""About 100 different proteins contribute to centrosomal functions including the products of (tubC), (spc97), (spc98), (cenA), (lis1), (nek2), (eb1), (cepJ) and cepG..."", … ""About 100 different proteins contribute to centrosomal functions including the products of tubC, (spc97), spc98, (cenA), (lis1), (nek2), (eb1), (cepJ) and (cepG)..""]"
…,…,…,…
"""these results suggest the exis…",2,2,"[""These results suggest the existence of an unidentified sulfated factor playing a key role in the intracellular killing of Klebsiella bacteria.."", ""These results suggest the existence of an unidentified sulfated factor playing a key role in the intracellular killing of Klebsiella bacteria.""]"
"""the sevencalcium up regulated …",2,2,"[""The sevencalcium up-regulated cup genes identified are: cupA, cupB, cupC, cupD, cupE, cupF, and cupG."", ""The sevencalcium up-regulated cup genes identified are: cupA, cupB, cupC, cupD, cupE, cupF, and cupG..""]"
"""an increase in ubiquitin posit…",2,2,"[""An increase in ubiquitin-positive protein aggregates with a corresponding decrease in proteasomal activity in these mutants reveals that the ubiquitin proteasome system (UPS) is dependent on intact autophagy for full activity (."", ""An increase in ubiquitin-positive protein aggregates with a corresponding decrease in proteasomal activity in these mutants reveals that the ubiquitin proteasome system (UPS) is dependent on intact autophagy for full activity. (.""]"


yes, there are actually quite many near duplicates. shall be also merged

In [340]:
# 2) Create a new merged claim_id based on claim_key
tmp = tmp.with_columns(
    claim_id_new=pl.col("claim_key").rank(method="dense").cast(pl.Int64)
)

# 3) Merge rows safely and keep your column order
#    - choose a canonical claim_plain (first seen)
#    - merge gene_id as union joined by ","
claim_cleaned = (
    tmp
    .group_by([
        "claim_id_new",
        "publication_id",
        "citation_captions",
        "anchor_pos",
        "year",
    ])
    .agg([
        pl.first("claim_plain").alias("claim_plain"),
        pl.first("anchors").alias("anchors"),
        pl.col("gene_id").unique().sort().str.join(",").alias("gene_id"),
    ])
    .select([
        pl.col("claim_id_new").alias("claim_id"),
        "claim_plain",
        "anchors",
        "publication_id",
        "citation_captions",
        "gene_id",
        "anchor_pos",
        "year",
    ])
    .sort(["claim_id", "publication_id", "anchor_pos"])
)

claim_cleaned


claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year
i64,str,list[struct[2]],i64,str,str,i64,i32
1,"""A basic region in the tail is …","[{94,[13954]}]",13954,"""Brzeska et al. 2014""","""DDB_G0289117""",94,2014
2,"""A cDNA clone derived from psvA…","[{102,[8316]}]",8316,"""Barklis and Lodish 1983""","""DDB_G0276869""",102,1983
3,"""A cDNA clone of this gene, D19…","[{89,[8283]}]",8283,"""Chisholm et al. 1984""","""DDB_G0267412""",89,1984
4,"""A co-immunoprecipitation assay…","[{158,[19729]}]",19729,"""Li et al. 2020""","""DDB_G0274607""",158,2020
5,"""A comparison of the genes tran…","[{185,[13151]}]",13151,"""Galardi-Castilla et al. 2013""","""DDB_G0268920""",185,2013
…,…,…,…,…,…,…,…
2059,"""Yet another role for NDP kinas…","[{131,[4899]}]",4899,"""Sonnemann and Mutzel 1995""","""DDB_G0273069,DDB_G0273805""",131,1995
2060,"""zizA null mutant cells do not …","[{72,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0275035""",72,2012
2061,"""ZizB also interacts with sever…","[{106,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0267374,DDB_G0272104""",106,2012


In [341]:
year_summary = (
    claim_cleaned
    .with_columns(pl.col("year").fill_null(-1).alias("year2"))
    .group_by("year2")
    .agg(pl.len().alias("n"))
    .sort("year2")
    .with_columns(
        pl.when(pl.col("year2") == -1).then(None).otherwise(pl.col("year2")).alias("year")
    )
    .select(["year", "n"])
)

year_summary


year,n
i32,u32
null,7
1956,1
1965,1
1967,3
1968,1
…,…
2016,76
2017,58
2018,81


In [342]:
claim_cleaned.write_parquet("output/cleaned/claim_cleaned_long.parquet")

In [343]:
claim_cleaned.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)


n_unique_claim_id
u32
2063


In [344]:
(
    claim_cleaned
    .drop("anchors")
    .write_csv("output/cleaned/claim_cleaned_long.tsv", separator="\t")
)


I still see some duplicated claims. There are still a few edge cases. But at this point, I am not gonna trying to “perfectly” clean. I will try to pick up a golden set out of it.

In [319]:
# manual inspection, will pick some golden set maunally later
claim_cleaned_manual = claim_cleaned.filter(pl.col("anchor_pos") != 1)
# claim_cleaned_gold=...

# match pmid

In [345]:
pub_pmid = pl.read_csv("output/publication_id_pmid.csv")
pub_pmid

publication_id,pmid
i64,i64
12,21243421
13,21239624
15,21235525
17,20950684
20,21150268
…,…
19689,32769116
19708,21551065
19728,32821814


In [346]:
# check if all publcation id can be mapped to pmid

# pick the right df names
a = claim_cleaned
b = pub_pmid

# make sure both publication_id columns are the same type (I recommend Int64)
a_ids = a.select(pl.col("publication_id").cast(pl.Int64)).unique()
b_ids = b.select(pl.col("publication_id").cast(pl.Int64)).unique()

# overlap + only-in sets
overlap = a_ids.join(b_ids, on="publication_id", how="inner")
only_a  = a_ids.join(b_ids, on="publication_id", how="anti")
only_b  = b_ids.join(a_ids, on="publication_id", how="anti")

summary = pl.DataFrame({
    "set": ["claim_cleaned", "pub_pmid", "overlap", "only_claim_cleaned", "only_pub_pmid"],
    "unique_count": [a_ids.height, b_ids.height, overlap.height, only_a.height, only_b.height],
})

summary

set,unique_count
str,i64
"""claim_cleaned""",1400
"""pub_pmid""",4341
"""overlap""",1373
"""only_claim_cleaned""",27
"""only_pub_pmid""",2968


27 ids are not mapped.

In [347]:
claim_cleaned_pmid = (
    claim_cleaned
    .with_columns(pl.col("publication_id").cast(pl.Int64))
    .join(pub_pmid, on="publication_id", how="left")
    .with_columns(
        pl.col("pmid").fill_null("NA")  # or keep as null if you prefer
    )
)

claim_cleaned_pmid

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid
i64,str,list[struct[2]],i64,str,str,i64,i32,str
1,"""A basic region in the tail is …","[{94,[13954]}]",13954,"""Brzeska et al. 2014""","""DDB_G0289117""",94,2014,"""24747353"""
2,"""A cDNA clone derived from psvA…","[{102,[8316]}]",8316,"""Barklis and Lodish 1983""","""DDB_G0276869""",102,1983,"""6301681"""
3,"""A cDNA clone of this gene, D19…","[{89,[8283]}]",8283,"""Chisholm et al. 1984""","""DDB_G0267412""",89,1984,"""6429548"""
4,"""A co-immunoprecipitation assay…","[{158,[19729]}]",19729,"""Li et al. 2020""","""DDB_G0274607""",158,2020,"""32818671"""
5,"""A comparison of the genes tran…","[{185,[13151]}]",13151,"""Galardi-Castilla et al. 2013""","""DDB_G0268920""",185,2013,"""23577638"""
…,…,…,…,…,…,…,…,…
2059,"""Yet another role for NDP kinas…","[{131,[4899]}]",4899,"""Sonnemann and Mutzel 1995""","""DDB_G0273069,DDB_G0273805""",131,1995,"""7733916"""
2060,"""zizA null mutant cells do not …","[{72,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0275035""",72,2012,"""22366457"""
2061,"""ZizB also interacts with sever…","[{106,[12098]}]",12098,"""Pakes et al. 2012""","""DDB_G0267374,DDB_G0272104""",106,2012,"""22366457"""


In [348]:
claim_cleaned_pmid.write_parquet("output/cleaned/claim_cleaned_long_pmids.parquet")

claim_cleaned_pmid_nonNA = claim_cleaned_pmid.filter(
    pl.col("pmid").is_not_null() & (pl.col("pmid") != "NA")
)

claim_cleaned_pmid_nonNA.write_parquet("output/cleaned/claim_cleaned_long_pmids_nonNA.parquet")


In [349]:
claim_cleaned_pmid_nonNA.select(
    pl.col("claim_id").n_unique().alias("n_unique_claim_id")
)


n_unique_claim_id
u32
2040


# how many we claims having abstracts on EPMC

In [350]:
EPMC = pl.read_parquet("output/cleaned/articles_all_cleaned_abstract.parquet")

In [351]:
EPMC

pmid,pmcid,doi,year,title,journal,authors,abstract_clean,file
str,str,str,str,str,str,str,str,str
"""2654141""","""PMC2115546""","""10.1083/jcb.108.5.1751""","""1989""","""Centrin-mediated microtubule s…","""The Journal of cell biology""","""Sanders MA, Salisbury JL.""","""Chlamydomonas cells excise the…","""article_fetching/output/all_cl…"
"""39528565""","""PMC11555045""","""10.1038/s41467-024-54272-4""","""2024""","""Nuclear localization sequence …","""Nature communications""","""Lim YJ, Yoon YJ, Lee H, Choi G…","""Plant pathogens secrete nuclea…","""article_fetching/output/all_cl…"
"""6319129""",null,"""10.1111/j.1432-1033.1984.tb078…","""1984""","""Investigations on stimulation …","""European journal of biochemist…","""Scholübbers HG, van Knippenber…","""The ability of 24 systematical…","""article_fetching/output/all_cl…"
"""28740830""","""PMC5502327""","""10.3389/fonc.2017.00139""","""2017""","""Structure, Activity Regulation…","""Frontiers in oncology""","""Mammucari C, Gherardi G, Rizzu…","""Mitochondrial Ca2+ uptake play…","""article_fetching/output/all_cl…"
"""18814278""",null,"""10.1002/cm.20314""","""2008""","""Correlated waves of actin fila…","""Cell motility and the cytoskel…","""Asano Y, Nagasaki A, Uyeda TQ.""","""Chemotaxis-deficient amiB-null…","""article_fetching/output/all_cl…"
…,…,…,…,…,…,…,…,…
"""36543032""","""PMC9889102""","""10.1016/j.ejmech.2022.115008""","""2023""","""Isoform selectivities of novel…","""European journal of medicinal …","""Smith JD, Brawley J, Bordenave…","""Muscle myosin inhibition could…","""article_fetching/output/all_cl…"
"""20023070""","""PMC2823002""","""10.1128/ec.00220-09""","""2010""","""Distinct subcellular localizat…","""Eukaryotic cell""","""Schilde C, Schönemann B, Sehri…","""We have identified new synapto…","""article_fetching/output/all_cl…"
"""10706824""",null,"""10.1006/scdb.1999.0343""","""1999""","""Control of spatial patterning …","""Seminars in cell & development…","""Mohanty S, Firtel RA.""","""The spatial patterning of pres…","""article_fetching/output/all_cl…"


In [352]:
a = claim_cleaned_pmid_nonNA.select(pl.col("pmid").cast(pl.Utf8).alias("pmid")).unique()
b = EPMC.select(pl.col("pmid").cast(pl.Utf8).alias("pmid")).unique()

overlap = a.join(b, on="pmid", how="inner")
only_a  = a.join(b, on="pmid", how="anti")
only_b  = b.join(a, on="pmid", how="anti")

summary = pl.DataFrame({
    "set": ["claim_cleaned_pmid_nonNA", "epmc_df", "overlap", "only_claim_cleaned_pmid", "only_epmc_df"],
    "unique_count": [a.height, b.height, overlap.height, only_a.height, only_b.height],
})

summary

set,unique_count
str,i64
"""claim_cleaned_pmid_nonNA""",1372
"""epmc_df""",20447
"""overlap""",1340
"""only_claim_cleaned_pmid""",32
"""only_epmc_df""",19107


In [353]:
only_claim_pmids = only_a.get_column("pmid")

rows_only_claim = claim_cleaned_pmid.filter(
    pl.col("pmid").cast(pl.Utf8).is_in(only_claim_pmids)
)

rows_only_claim

/var/folders/yr/ljjqmpj92wncw9wrv495wq300000gn/T/ipykernel_50959/496506703.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  rows_only_claim = claim_cleaned_pmid.filter(


claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid
i64,str,list[struct[2]],i64,str,str,i64,i32,str
98,"""Adenylyl cyclase increases mor…","[{133,[9781]}, {136,[10027]}, {139,[9617]}]",9781,"""Roos et al. 1977""","""DDB_G0281545""",133,1977,"""191758"""
98,"""Adenylyl cyclase increases mor…","[{133,[9781]}, {136,[10027]}, {139,[9617]}]",10027,"""Klein 1976""","""DDB_G0281545""",136,1976,"""183985"""
200,"""axeB was identified as a mutan…","[{139,[10264]}, {142,[7409]}]",10264,"""Williams et al. 1974""","""DDB_G0350652""",139,1974,"""4474353"""
206,"""Biochemical assays found a 10 …","[{114,[10751]}]",10751,"""Ashworth and Sussman, 1967""","""DDB_G0289875""",114,1967,"""6067273"""
207,"""Biochemical assays found a 10 …","[{126,[10751]}]",10751,"""Ashworth and Sussman, 1967""","""DDB_G0277879""",126,1967,"""6067273"""
…,…,…,…,…,…,…,…,…
1661,"""The major spore coat proteins,…","[{172,[9387]}, {173,[8541]}, … {176,[6031]}]",6653,"""Fosnaugh and Loomis, 1989""","""DDB_G0276941""",175,1989,"""2587278"""
1661,"""The major spore coat proteins,…","[{172,[9387]}, {173,[8541]}, … {176,[6031]}]",9387,"""Orlowski and Loomis, 1979""","""DDB_G0276761,DDB_G0277141,DDB_…",172,1979,"""499661"""
1783,"""The specific activity of glyco…","[{201,[10721]}, {202,[9996]}]",10721,"""Wright and Dalhberg, 1967""","""DDB_G0267674""",201,1967,"""6069170"""


Some claims are not covered. It seems that most of them are old paper that there are no abstract available. So I remove those and merge title and abstract to the claim table

In [354]:
epmc_small = EPMC.select([
    pl.col("pmid").cast(pl.Utf8).alias("pmid"),
    pl.col("title").alias("title"),
    pl.col("abstract_clean").alias("abstract_clean"),
])

claim_small = claim_cleaned_pmid_nonNA.with_columns(
    pl.col("pmid").cast(pl.Utf8).alias("pmid")
)

# inner join = ignore only-claim PMIDs
claim_cleaned_pmid_nonNA_abstract = claim_small.join(epmc_small, on="pmid", how="inner")

claim_cleaned_pmid_nonNA_abstract

claim_id,claim_plain,anchors,publication_id,citation_captions,gene_id,anchor_pos,year,pmid,title,abstract_clean
i64,str,list[struct[2]],i64,str,str,i64,i32,str,str,str
1878,"""These results suggest that Atg…","[{175,[16288]}]",16288,"""Messling et al. 2017""","""DDB_G0286191,DDB_G0290491""",175,2017,"""28413119""","""The two Dictyostelium discoide…","""Autophagy is a highly conserve…"
403,"""Disruption of csaA results in …","[{103,[6630]}]",6630,"""Harloff et al. 1989""","""DDB_G0289073""",103,1989,"""2515990""","""Selective elimination of the c…","""The contact site A glycoprotei…"
1105,"""MyoB is non-filamentous and co…","[{163,[6608]}, {164,[6857]}, {165,[12097]}]",12097,"""Brzeska et al. 2012""","""DDB_G0289117""",165,2012,"""22367211""","""Molecular basis of dynamic rel…","""Class I myosins have a single …"
1196,"""pestis recruits host Rab1B pro…","[{235,[17146]}]",17146,"""Markman et al. 2018""","""DDB_G0277867""",235,2018,"""29350155""","""Yersinia pestis Survival and R…","""Plague ecology is characterize…"
582,"""FspA protein is an essential c…","[{122,[13550]}]",13550,"""Lima et al. 2013""","""DDB_G0277237""",122,2013,"""24128258""","""Two distinct sensing pathways …","""Recognition of bacteria by met…"
…,…,…,…,…,…,…,…,…,…,…
439,"""Dominant negative rabS cells d…","[{174,[17491]}]",17491,"""Yarbrough et al. 2018""","""DDB_G0282537""",174,2018,"""29843387""","""The Effect of Overexpressed Dd…","""Rab GTPases are essential regu…"
1309,"""Rab2A is associated with both …","[{83,[15422]}, {136,[17491]}]",17491,"""Yarbrough et al. 2018""","""DDB_G0282537""",136,2018,"""29843387""","""The Effect of Overexpressed Dd…","""Rab GTPases are essential regu…"
516,"""Expression of cotE is dependen…","[{169,[11602]}]",11602,"""Huang et al. 2011""","""DDB_G0277903""",169,2011,"""21810415""","""BzpF is a CREB-like transcript…","""The cAMP response element-bind…"


In [357]:
claim_cleaned_pmid_nonNA_abstract.write_parquet("output/cleaned/claim_cleaned_long_pmids_nonNA_abstract.parquet")
